# 05 — Graph Reinforcement Learning: Coloring and Navigation

**Goal:** Train an RL agent to make sequential decisions on a graph.
Compare random/greedy baselines with a learning algorithm (DQN).

**TGraphX subsystem:** `tgraphx.rl`

**Data:** Tiny synthetic graph environments.

**Runtime:** < 60 seconds on CPU.

**Honest note:** TGraphX RL is a research-focused foundation, not a
production RLlib/SB3 replacement.  These demonstrations show the API and
learning dynamics on small graphs.

In [ ]:
from tgraphx import run_graph_rl, list_graph_rl_algorithms
from tgraphx.rl import EarlyStoppingCallback, CSVLoggerCallback
import tempfile, json
print("Available RL algorithms:")
for name, desc in list_graph_rl_algorithms().items():
    print(f"  {name}: {desc}")

## 2. Baseline Comparison: Random vs Greedy vs DQN

In [ ]:
results = {}
for algo in ["random", "greedy", "dqn"]:
    r = run_graph_rl(
        env="graph_navigation",
        algorithm=algo,
        episodes=30,
        seed=42,
    )
    results[algo] = r.metrics["mean_return"]
    print(f"{algo:8s}: mean_return={r.metrics['mean_return']:.2f}")

## 3. DQN with Early Stopping Callback

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    csv_log = CSVLoggerCallback(tmpdir + "/episodes.csv")
    stopper = EarlyStoppingCallback(monitor="reward", patience=8, mode="max")

    r = run_graph_rl(
        env="graph_navigation",
        algorithm="dqn",
        episodes=50,
        seed=42,
        callbacks=[csv_log, stopper],
    )

    print(f"Stopped early: {getattr(r, 'stopped_early', False)}")
    print(f"Mean return: {r.metrics['mean_return']:.2f}")
    # Print a few CSV rows.
    import csv
    with open(tmpdir + "/episodes.csv") as f:
        rows = list(csv.DictReader(f))
    print(f"Logged {len(rows)} episodes.")
    if rows:
        print("Sample:", rows[-1])

## 4. Graph Coloring Environment

In [ ]:
# Try a graph coloring environment: agent assigns colors to nodes.
r_coloring = run_graph_rl(
    env="graph_coloring",
    algorithm="random",
    episodes=10,
    seed=0,
)
print("Coloring env mean return:", r_coloring.metrics["mean_return"])
print("Config:", r_coloring.config)

## 5. Key Concepts

| Concept | In TGraphX |
|---|---|
| Environment | `GraphEnv` subclass (navigation, coloring, max-cut, …) |
| Observation | `{"node_features": Tensor, "edge_index": Tensor, ...}` |
| Action | Discrete node/edge choice or continuous vector |
| Policy | `GraphPolicyNetwork` (graph-based forward pass) |
| Algorithms | Random, Greedy, REINFORCE, A2C, DQN, Double DQN, PPO, TD3, SAC |
| Callbacks | `EarlyStoppingCallback`, `CSVLoggerCallback` |

## 6. Next Steps
- **Tutorial:** `tutorials/graph_rl_quickstart.py`
- **Algorithms doc:** `docs/graph_rl_algorithms.md`
- **Limitations:** DQN on navigation converges slowly; PPO/SAC need more
  tuning.  For serious RL research, combine these foundations with a
  dedicated framework.